# SC-22-Solana-Anchor - Solana avec Anchor

[<< Move & Sui](SC-21-Move-Sui.ipynb) | [Cross-Chain >>](../06-Real-World/SC-23-Cross-Chain.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre l'architecture **Solana** (accounts, programs)
2. Utiliser le framework **Anchor** (Rust)
3. Creer des **PDAs** (Program Derived Addresses)
4. Implementer des **CPIs** (Cross-Program Invocations)

### Prerequis

- [SC-1](../01-Solidity-Foundation/SC-3-Solidity-Basics.ipynb) a [SC-4](../01-Solidity-Foundation/SC-6-Errors-Events.ipynb) completes (concepts)
- Notions de base en Rust (variables, ownership, structs)
- Solana CLI et Anchor installe (voir setup dans le notebook)

### Duree estimee : 55 minutes

---

## 1. Architecture Solana

Solana utilise un modele different d'Ethereum.

In [1]:
# Comparaison Ethereum vs Solana
print("""
ETHEREUM vs SOLANA

| Concept       | Ethereum              | Solana                  |
|--------------|----------------------|-------------------------|
| Execution    | EVM                   | BPF (Berkeley Packet Filter) |
| State        | Storage in contract  | Accounts (separates)    |
| Langage      | Solidity              | Rust (via Anchor)       |
| Adressage    | Nonces sequentielles  | Hash d'accounts         |
| Fees         | Gas (variable)        | Compute units + rent    |
| TPS          | ~15-30                | ~65,000                 |

COMPOSANTS SOLANA:
- Program    : Code executable (stateless)
- Account    : Stockage de donnees (state)
- PDA        : Program Derived Address
- CPI        : Cross-Program Invocation
- Token      : SPL Token (equivalent ERC-20/721)
""")


ETHEREUM vs SOLANA

| Concept       | Ethereum              | Solana                  |
|--------------|----------------------|-------------------------|
| Execution    | EVM                   | BPF (Berkeley Packet Filter) |
| State        | Storage in contract  | Accounts (separates)    |
| Langage      | Solidity              | Rust (via Anchor)       |
| Adressage    | Nonces sequentielles  | Hash d'accounts         |
| Fees         | Gas (variable)        | Compute units + rent    |
| TPS          | ~15-30                | ~65,000                 |

COMPOSANTS SOLANA:
- Program    : Code executable (stateless)
- Account    : Stockage de donnees (state)
- PDA        : Program Derived Address
- CPI        : Cross-Program Invocation
- Token      : SPL Token (equivalent ERC-20/721)



---

## 2. Programme Anchor Simple

In [2]:
# Counter en Anchor
ANCHOR_COUNTER = '''
// Fichier: programs/counter/src/lib.rs

use anchor_lang::prelude::*;

declare_id!("Counter11111111111111111111111111111111111");

#[program]
pub mod counter {
    use super::*;

    pub fn initialize(ctx: Context<Initialize>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count = 0;
        counter.authority = ctx.accounts.authority.key();
        Ok(())
    }

    pub fn increment(ctx: Context<Increment>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count += 1;
        Ok(())
    }

    pub fn decrement(ctx: Context<Decrement>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        require!(counter.count > 0, CounterError::Underflow);
        counter.count -= 1;
        Ok(())
    }
}

#[derive(Accounts)]
pub struct Initialize<'info> {
    #[account(
        init,
        payer = authority,
        space = 8 + Counter::INIT_SPACE,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    #[account(mut)]
    pub authority: Signer<'info>,
    pub system_program: Program<'info, System>,
}

#[derive(Accounts)]
pub struct Increment<'info> {
    #[account(
        mut,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    pub authority: Signer<'info>,
}

#[derive(Accounts)]
pub struct Decrement<'info> {
    #[account(
        mut,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    pub authority: Signer<'info>,
}

#[account]
#[derive(InitSpace)]
pub struct Counter {
    pub authority: Pubkey,
    pub count: u64,
}

#[error_code]
pub enum CounterError {
    #[msg("Counter cannot go below zero")]
    Underflow,
}
'''

print("Programme Counter Anchor:")
print(ANCHOR_COUNTER)

Programme Counter Anchor:

// Fichier: programs/counter/src/lib.rs

use anchor_lang::prelude::*;

declare_id!("Counter11111111111111111111111111111111111");

#[program]
pub mod counter {
    use super::*;

    pub fn initialize(ctx: Context<Initialize>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count = 0;
        counter.authority = ctx.accounts.authority.key();
        Ok(())
    }

    pub fn increment(ctx: Context<Increment>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count += 1;
        Ok(())
    }

    pub fn decrement(ctx: Context<Decrement>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        require!(counter.count > 0, CounterError::Underflow);
        counter.count -= 1;
        Ok(())
    }
}

#[derive(Accounts)]
pub struct Initialize<'info> {
    #[account(
        init,
        payer = authority,
        space = 8 + Counter::INIT_SPACE,
        seeds = [b"counter", authority.k

### Interpretation : Architecture Solana vs Ethereum

Ce tableau compare les differents modeles d'execution et de stockage entre Ethereum et Solana.

| Concept | Ethereum | Solana | Impact pratique |
|---------|----------|---------|-----------------|
| **Execution** | EVM (bytecode stack-based) | BPF (Berkeley Packet Filter, compile depuis Rust/C++) | Solana est plus rapide (JIT, native compilation) |
| **State** | Stocke dans le contrat (storage) | Comptes separes du programme | Solana : plus de parallelisme, pas de "global state" |
| **Langage** | Solidity (type, haut niveau) | Rust (systeme, verifie) | Rust est plus strict (ownership) mais plus performant |
| **Adressage** | Nonces sequentielles (contract creation count) | Hash des seeds du PDA | Solana : addresses deterministes, previsibles |
| **Fees** | Gas (variable selon complexite) | Compute units + rent exemption | Solana : couts predictibles, rent pour stockage durable |
| **TPS** | ~15-30 (Layer 1) | ~65,000 (theorique) | Solana : haut debit, sharding natif |

**Points cles** :
1. **Programs stateless** : Le code est separe des donnees (comptes). Le programme ne contient pas d'etat.
2. **Accounts model** : Chaque compte est une entite separee avec son propre solde (Lamports) et ses donnees.
3. **Rent exemption** : Les comptes doivent avoir un solde minimum pour exister (rent). Au-dessus de ce seuil, pas de frais de stockage.
4. **Parallelisme** : Solana peut executer plusieurs transactions en parallele si elles touchent des comptes differents.

**Implications pour les developpeurs** :
- Sur Ethereum : on deploye un contrat avec tout l'etat (mappings, arrays)
- Sur Solana : on deploye un programme (code) et on cree des comptes PDAs pour chaque utilisateur/entite
- Le modele Solana permet plus de parallelisme mais demande plus de planification (structure des comptes)

> **Note technique** : BPF (Berkeley Packet Filter) est un format bytecode securise qui s'execute dans le noyau Linux. Solana l'utilise pour la securite et la performance (JIT compilation vers machine code).


---

## 3. PDAs (Program Derived Addresses)

Les PDAs sont des adresses derivees deterministes.

In [3]:
# PDA explanation
print("""
PDA (Program Derived Address)

PROPRIETES:
- Adresse derivee du program_id et de seeds
- Pas de cle privee (programme seulement)
- Deterministe: meme seeds = meme PDA
- Bump: [b"seed", bump] pour trouver le PDA

GENERATION:
seeds = [b"counter", authority.key().as_ref()]
(pda, bump) = Pubkey::find_program_address(&seeds, program_id)

UTILISATION:
1. Stockage de donnees utilisateur
2. Escrows
3. Token accounts
4. Metadata

EXEMPLE DANS ANCHOR:
#[account(
    seeds = [b"counter", authority.key().as_ref()],
    bump
)]
pub counter: Account<'info, Counter>,
""")


PDA (Program Derived Address)

PROPRIETES:
- Adresse derivee du program_id et de seeds
- Pas de cle privee (programme seulement)
- Deterministe: meme seeds = meme PDA
- Bump: [b"seed", bump] pour trouver le PDA

GENERATION:
seeds = [b"counter", authority.key().as_ref()]
(pda, bump) = Pubkey::find_program_address(&seeds, program_id)

UTILISATION:
1. Stockage de donnees utilisateur
2. Escrows
3. Token accounts
4. Metadata

EXEMPLE DANS ANCHOR:
#[account(
    seeds = [b"counter", authority.key().as_ref()],
    bump
)]
pub counter: Account<'info, Counter>,



### Interpretation : Programme Counter Anchor

Ce code presente un programme de compteur simple avec les concepts fondamentaux d'Anchor.

| Composant | Role | Details |
|-----------|------|---------|
| `#[program]` | Module de fonctions publiques | Points d'entree du programme (appelables par les clients) |
| `Context<Initialize>` | Validation des comptes | Anchor verifie que tous les comptes passes sont valides |
| `#[account(init)]` | Creation de compte | Alloue un nouveau compte PDA avec `init` + `payer` + `space` + `seeds` |
| `#[account(mut)]` | Compte modifiable | Le compte peut etre modifie pendant la transaction |
| `#[error_code]` | Erreurs custom | Erreurs specifiques au programme (ici: `Underflow`) |

**Points cles** :
1. `initialize` cree un PDA `counter` avec seeds `[b"counter", authority.key().as_ref()]` → 1 counter par utilisateur
2. `space = 8 + Counter::INIT_SPACE` : 8 octets de discriminant Anchor + taille de la struct `Counter` (32 + 8 = 40)
3. `increment` et `decrement` utilisent les memes seeds pour retrouver le PDA du counter
4. `require!(counter.count > 0, CounterError::Underflow)` : protection contre underflow (custom error)

**Comparaison Solidity vs Anchor** :
- **Solidity** : `mapping(address => uint256) public counters;` (stocke dans le contrat)
- **Anchor** : Chaque utilisateur a son propre compte PDA `Counter` (separe du programme)

**Avantages du modele Solana** :
- Parallelelisme : chaque utilisateur modifie son propre compte (pas de contention)
- Rent exemption : l'utilisateur paie pour son propre compte (pas le deployeur)
- Flexibilite : les comptes peuvent etre fermes pour recuperer la rent

> **Note technique** : Le discriminant Anchor (8 octets) est un hash unique qui identifie le type de compte. Il permet a Anchor de verifier qu'un compte passe est bien du type attendu.


### Exercice : Generateur de struct Anchor

Ecrire un programme Anchor pour gerer un profil utilisateur sur Solana. Completez la fonction `generate_anchor_profile` qui produit le code Rust du programme.

**Objectifs** :
1. Definir une struct `UserProfile` avec les champs : owner (Pubkey), username (String[32]), bio (String[280]), reputation (u64)
2. Implementer les fonctions `create_profile` et `update_reputation`
3. Calculer l'espace necessaire pour le compte

**Indices** :
- Utiliser `#[account(init, payer = owner, space = 8 + UserProfile::INIT_SPACE, seeds = [...], bump)]` pour la creation
- Le PDA est derive de `b"profile"` et de l'adresse du proprietaire
- `update_reputation` prend un montant positif ou negatif (utiliser `i64` pour l'argument)

In [4]:
def generate_anchor_profile() -> str:
    """Generer le code Rust d'un programme Anchor UserProfile.

    Le programme doit definir :
    - Struct UserProfile : owner (Pubkey), username (String[32]), bio (String[280]), reputation (u64)
    - Fonction create_profile : initialise le profil avec username et bio
    - Fonction update_reputation : ajoute un delta (positif ou negatif) a la reputation

    Retourner le code Rust sous forme de chaine.

    TODO etudiant : implementez generate_anchor_profile.
    Indice : inspirez-vous du programme Counter vu en section 2.
    Les seeds du PDA seront [b"profile", owner.key().as_ref()].
    """
    # TODO etudiant : ecrivez le code Rust complet
    return ""  # TODO etudiant : remplacez par le code du programme


# Test : afficher le code genere
code = generate_anchor_profile()
if code:
    print("Programme UserProfile Anchor :")
    print(code)
else:
    print("Exercice a completer : generate_anchor_profile")

Exercice a completer : generate_anchor_profile


---

## 4. CPI (Cross-Program Invocation)

In [5]:
# CPI example
CPI_EXAMPLE = '''
// CPI vers System Program pour transfer SOL
use anchor_lang::system_program;

pub fn transfer_sol(ctx: Context<TransferSol>, amount: u64) -> Result<()> {
    let cpi_context = CpiContext::new(
        ctx.accounts.system_program.to_account_info(),
        system_program::Transfer {
            from: ctx.accounts.from.to_account_info(),
            to: ctx.accounts.to.to_account_info(),
        }
    );
    system_program::transfer(cpi_context, amount)
}

#[derive(Accounts)]
pub struct TransferSol<'info> {
    #[account(mut)]
    pub from: Signer<'info>,
    #[account(mut)]
    pub to: AccountInfo<'info>,
    pub system_program: Program<'info, System>,
}

// CPI vers Token Program
use anchor_spl::token::{self, Transfer as TokenTransfer};

pub fn transfer_token(
    ctx: Context<TransferToken>,
    amount: u64
) -> Result<()> {
    let cpi_accounts = TokenTransfer {
        from: ctx.accounts.from.to_account_info(),
        to: ctx.accounts.to.to_account_info(),
        authority: ctx.accounts.authority.to_account_info(),
    };
    let cpi_program = ctx.accounts.token_program.to_account_info();
    token::transfer(CpiContext::new(cpi_program, cpi_accounts), amount)
}
'''

print("CPI Examples:")
print(CPI_EXAMPLE)

CPI Examples:

// CPI vers System Program pour transfer SOL
use anchor_lang::system_program;

pub fn transfer_sol(ctx: Context<TransferSol>, amount: u64) -> Result<()> {
    let cpi_context = CpiContext::new(
        ctx.accounts.system_program.to_account_info(),
        system_program::Transfer {
            from: ctx.accounts.from.to_account_info(),
            to: ctx.accounts.to.to_account_info(),
        }
    );
    system_program::transfer(cpi_context, amount)
}

#[derive(Accounts)]
pub struct TransferSol<'info> {
    #[account(mut)]
    pub from: Signer<'info>,
    #[account(mut)]
    pub to: AccountInfo<'info>,
    pub system_program: Program<'info, System>,
}

// CPI vers Token Program
use anchor_spl::token::{self, Transfer as TokenTransfer};

pub fn transfer_token(
    ctx: Context<TransferToken>,
    amount: u64
) -> Result<()> {
    let cpi_accounts = TokenTransfer {
        from: ctx.accounts.from.to_account_info(),
        to: ctx.accounts.to.to_account_info(),
        a

### Interpretation : PDAs (Program Derived Addresses)

Les PDAs sont une innovation unique de Solana : des adresses sans cle privee, controlees par des programmes.

| Propriete | Description | Impact |
|-----------|-------------|--------|
| **Pas de cle privee** | Le PDA est derive de `program_id + seeds` | Seul le programme peut signer pour le PDA |
| **Deterministe** | Memes seeds = meme PDA (toujours) | Pas de collision, pas de generation aleatoire |
| **Bump canonique** | Trouve le premier `bump` (1-255) qui donne une adresse hors courbe ed25519 | Garantit que seul le programme peut signer |

**Points cles** :
1. `seeds = [b"counter", authority.key().as_ref()]` : le PDA est unique par utilisateur
2. Le `bump` est la valeur qui fait que l'adresse est "hors courbe" (pas de cle privee correspondante)
3. Anchor calcule automatiquement le bump avec `#[account(seeds = [...], bump)]`
4. Les PDAs sont utilises pour : comptes de donnees utilisateur, vaults, escrows, metadata NFT

**Cas d'usage typiques** :
- **Compte utilisateur** : `seeds = [b"user", user_pubkey.as_ref()]` → 1 compte par utilisateur
- **Vault d'escrow** : `seeds = [b"vault", escrow_id.as_ref()]` → vault unique par escrow
- **Metadata NFT** : `seeds = [b"metadata", mint.as_ref()]` → metadata unique par mint

**Generation manuelle** (Rust) :
```rust
let (pda, bump) = Pubkey::find_program_address(
    &[b"counter", authority.key().as_ref()],
    program_id
);
```

> **Note technique** : Le "bump" est entre 1 et 255. Si 255 ne suffit pas, il n'y a pas de PDA valide. Anchor utilise le "bump canonique" (le premier qui marche) pour garantir l'unicite.


### Exercice : Calculateur de PDA et espace de compte

Sur Solana, chaque compte doit allouer un espace de stockage payant (rent). La taille depend des champs de la struct Anchor. Completez les fonctions `calculate_pda` et `calculate_account_size`.

**Objectifs** :
1. Deriver une PDA a partir de seeds et d'un program_id (simulation)
2. Calculer l'espace necessaire pour un compte Anchor (discriminant 8 octets + taille des champs)
3. Estimer le cout en SOL (rent) pour creer le compte

**Indices** :
- L'espace Anchor = 8 octets (discriminant) + somme des tailles des champs
- Types Anchor : Pubkey = 32 octets, u64 = 8 octets, bool = 1 octet, String = 4 + longueur, Vec<T> = 4 + N * taille(T)
- Le rent sur Solana est environ 0.000890880 SOL par octet (a la creation)

In [6]:
import hashlib


# Tailles des types Anchor (en octets)
ANCHOR_TYPE_SIZES = {
    "Pubkey": 32, "u64": 8, "u32": 4, "u16": 2, "u8": 1,
    "i64": 8, "i32": 4, "bool": 1, "f64": 8,
}

RENT_PER_BYTE = 0.000890880  # SOL par octet (approximation Solana)


def calculate_pda(seeds: list, program_id: str) -> str:
    """Simuler la derivation d'une PDA Solana.

    La vraie PDA utilise ed25519 off-curve. Ici on simule avec SHA-256.

    TODO etudiant :
    Etape 1 : concatener les seeds et le program_id
    Etape 2 : hasher avec SHA-256
    Etape 3 : retourner le hash hexadecimal comme "PDA simulee"

    Args:
        seeds: liste de bytes ou strings (les seeds du PDA)
        program_id: adresse du programme (hex string)

    Returns:
        str : la PDA simulee (hash hex)
    """
    # TODO etudiant : implementez calculate_pda
    return ""  # TODO etudiant


def calculate_account_size(fields: list) -> int:
    """Calculer l'espace necessaire pour un compte Anchor.

    Args:
        fields: liste de tuples (nom_du_champ, type_du_champ, taille_optionnelle)
            ex: [("authority", "Pubkey", None), ("count", "u64", None), ("name", "String", 32)]

    Returns:
        int : taille totale en octets (incluant le discriminant Anchor de 8 octets)

    TODO etudiant :
    Etape 1 : ajouter 8 octets pour le discriminant Anchor
    Etape 2 : pour chaque champ, ajouter la taille du type (utiliser ANCHOR_TYPE_SIZES)
    Etape 3 : pour String, ajouter 4 + taille_max ; pour Vec, ajouter 4 + N * taille_element
    """
    # TODO etudiant : implementez calculate_account_size
    return 0  # TODO etudiant


# Test : compte Counter avec authority + count
fields = [
    ("authority", "Pubkey", None),
    ("count", "u64", None),
]
size = calculate_account_size(fields)
pda = calculate_pda([b"counter", b"alice_pubkey"], "Counter11111...11111")
rent = size * RENT_PER_BYTE if size > 0 else 0
print(f"Espace compte : {size} octets")
print(f"PDA simulee : {pda[:32]}...")
print(f"Rent estime : {rent:.6f} SOL")
print("Exercice a completer : calculate_pda et calculate_account_size")

Espace compte : 0 octets
PDA simulee : ...
Rent estime : 0.000000 SOL
Exercice a completer : calculate_pda et calculate_account_size


---

## 5. Commandes Anchor CLI

In [7]:
# Anchor CLI commands
print("""
INSTALLATION:
cargo install --git https://github.com/coral-xyz/anchor avm --locked --force
avm install latest
avm use latest

CREATION PROJET:
anchor init my_program
cd my_program

STRUCTURE:
my_program/
|-- Anchor.toml          # Configuration
|-- Cargo.toml
|-- programs/
|   `-- my_program/
|       |-- Cargo.toml
|       `-- src/
|           `-- lib.rs   # Code principal
|-- tests/
|   `-- my_program.ts    # Tests
`-- app/                 # Frontend

COMMANDES:
anchor build              # Compiler
anchor test               # Executer les tests
anchor deploy             # Deploier
anchor run <script>       # Executer un script

CLUSTER:
anchor test --skip-local-validator  # Avec validator externe
anchor deploy --provider.cluster devnet
""")


INSTALLATION:
cargo install --git https://github.com/coral-xyz/anchor avm --locked --force
avm install latest
avm use latest

CREATION PROJET:
anchor init my_program
cd my_program

STRUCTURE:
my_program/
|-- Anchor.toml          # Configuration
|-- Cargo.toml
|-- programs/
|   `-- my_program/
|       |-- Cargo.toml
|       `-- src/
|           `-- lib.rs   # Code principal
|-- tests/
|   `-- my_program.ts    # Tests
`-- app/                 # Frontend

COMMANDES:
anchor build              # Compiler
anchor test               # Executer les tests
anchor deploy             # Deploier
anchor run <script>       # Executer un script

CLUSTER:
anchor test --skip-local-validator  # Avec validator externe
anchor deploy --provider.cluster devnet



### Interpretation : Cross-Program Invocations (CPI)

Les CPIs permettent a un programme Solana d'appeler un autre programme (equivalent des `delegatecall` en Solidity).

| Type de CPI | Programme cible | Operation |
|-------------|----------------|-----------|
| `system_program::transfer` | System Program | Transferts de SOL native |
| `token::transfer` | SPL Token Program | Transferts de tokens SPL |
| Metaplex CPI | Metaplex | Creation NFT, metadata |
| Custom CPI | Autres programmes utilisateur | Composition de fonctionnalites |

**Points cles** :
1. `CpiContext::new` cree un contexte CPI standard (signataire = l'appelant)
2. `CpiContext::new_with_signer` permet a un PDA de signer (cas du vault escrow)
3. Tous les comptes necessaires doivent etre passes dans la struct `Accounts` (Anchor verifie la coherence)
4. Les CPIs sont payables par l'appelant (compute units + rent exemption)

**Difference EVM vs Solana** :
- EVM : `delegatecall` execute le code d'un autre contrat DANS le contexte du contrat appelant
- Solana : CPI execute le code d'un autre programme AVEC ses propres comptes (contexte separe)

**Pattern CPI classique** :
1. Definir les comptes necessaires dans la struct `#[derive(Accounts)]`
2. Creer le `CpiContext` avec les bons comptes et autorites
3. Appeler la fonction du programme cible (ex: `token::transfer`)

> **Note technique** : Les CPIs sont COMPOSES sur Solana. Un programme A peut appeler B, qui appelle C, etc. La limite de profondeur est de 4 appels chaines.


### Exercice : Analyseur de structure de comptes Anchor

L'avantage principal de Solana sur Ethereum est le parallelisme des transactions : deux transactions qui modifient des comptes differents peuvent s'executer en parallele. Completez la fonction `analyze_parallelism` qui determine quelles transactions peuvent s'executer simultanement.

**Objectifs** :
1. Identifier les comptes en lecture et ecriture pour chaque instruction
2. Determiner quelles paires de transactions sont parallelisables
3. Calculer le throughput theorique (nombre de transactions executables en parallele)

**Indices** :
- Deux transactions sont parallelisables si elles n'ont aucun compte en ecriture en commun
- Un compte en lecture seule ne bloque pas le parallelisme
- Le throughput theorique = nombre maximum de transactions parallelisables dans un meme batch

In [8]:
def analyze_parallelism(transactions: list) -> dict:
    """Analyser le parallelisme possible entre transactions Solana.

    Args:
        transactions: liste de dict, chaque dict contient :
            - "name": nom de la transaction
            - "reads": set de comptes lus
            - "writes": set de comptes ecrits

    Returns:
        dict avec :
            - "parallelizable_pairs": liste de tuples (i, j) parallelisables
            - "conflicts": liste de tuples (i, j) en conflit
            - "max_parallel": nombre max de tx parallelisables
            - "sequential_only": True si aucune paire n'est parallelisable

    TODO etudiant : implementez analyze_parallelism.
    Etape 1 : pour chaque paire (i, j), verifier si writes[i] et writes[j] s'intersectent
    Etape 2 : si pas d'intersection des writes, la paire est parallelisable
    Etape 3 : calculer max_parallel (taille du plus grand sous-ensemble mutuellement parallelisable)
    """
    # TODO etudiant : implementez analyze_parallelism
    return {"parallelizable_pairs": [], "conflicts": [], "max_parallel": 0, "sequential_only": True}


# Exemple : transactions Solana typiques
txs = [
    {"name": "increment_counter_Alice", "reads": {"counter_alice"}, "writes": {"counter_alice"}},
    {"name": "increment_counter_Bob",   "reads": {"counter_bob"},   "writes": {"counter_bob"}},
    {"name": "transfer_SOL_Alice_to_Bob", "reads": {"alice_wallet", "bob_wallet"}, "writes": {"alice_wallet", "bob_wallet"}},
    {"name": "mint_token",              "reads": {"mint_authority"}, "writes": {"mint_authority", "token_account"}},
    {"name": "read_counter_Alice",      "reads": {"counter_alice"}, "writes": set()},
]

result = analyze_parallelism(txs)
print(f"Paires parallelisables : {len(result['parallelizable_pairs'])}")
print(f"Conflits : {len(result['conflicts'])}")
print(f"Max parallel : {result['max_parallel']}")
print(f"Sequential only : {result['sequential_only']}")
print("Exercice a completer : analyze_parallelism")

Paires parallelisables : 0
Conflits : 0
Max parallel : 0
Sequential only : True
Exercice a completer : analyze_parallelism


---

## 6. Exemples guidés

In [9]:
# Exercice: Token Escrow
EXERCISE_ESCROW = '''
#[program]
pub mod escrow {
    use super::*;

    pub fn make(
        ctx: Context<Make>,
        seed: u64,
        receive_amount: u64,
    ) -> Result<()> {
        let escrow = &mut ctx.accounts.escrow;
        escrow.seed = seed;
        escrow.initializer = ctx.accounts.initializer.key();
        escrow.mint_a = ctx.accounts.mint_a.key();
        escrow.mint_b = ctx.accounts.mint_b.key();
        escrow.receive_amount = receive_amount;

        // Transfer tokens to vault
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.deposit.to_account_info(),
                    to: ctx.accounts.vault.to_account_info(),
                    authority: ctx.accounts.initializer.to_account_info(),
                },
            ),
            ctx.accounts.deposit.amount,
        )
    }

    pub fn take(ctx: Context<Take>) -> Result<()> {
        // Transfer B to initializer
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.depositor_token_b.to_account_info(),
                    to: ctx.accounts.initializer_token_b.to_account_info(),
                    authority: ctx.accounts.depositor.to_account_info(),
                },
            ),
            ctx.accounts.escrow.receive_amount,
        )?;

        // Transfer A to depositor
        token::transfer(
            CpiContext::new_with_signer(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.vault.to_account_info(),
                    to: ctx.accounts.depositor_token_a.to_account_info(),
                    authority: ctx.accounts.escrow.to_account_info(),
                },
                &[&[
                    b"escrow",
                    ctx.accounts.maker.key().as_ref(),
                    &ctx.accounts.escrow.seed.to_le_bytes()[..],
                    &[ctx.bumps.escrow],
                ]],
            ),
            ctx.accounts.vault.amount,
        )
    }
}
'''
print("Exercice Escrow:")
print(EXERCISE_ESCROW)

Exercice Escrow:

#[program]
pub mod escrow {
    use super::*;

    pub fn make(
        ctx: Context<Make>,
        seed: u64,
        receive_amount: u64,
    ) -> Result<()> {
        let escrow = &mut ctx.accounts.escrow;
        escrow.seed = seed;
        escrow.initializer = ctx.accounts.initializer.key();
        escrow.mint_a = ctx.accounts.mint_a.key();
        escrow.mint_b = ctx.accounts.mint_b.key();
        escrow.receive_amount = receive_amount;

        // Transfer tokens to vault
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.deposit.to_account_info(),
                    to: ctx.accounts.vault.to_account_info(),
                    authority: ctx.accounts.initializer.to_account_info(),
                },
            ),
            ctx.accounts.deposit.amount,
        )
    }

    pub fn take(ctx: Context<Take>) -> Result<()> {


### Interpretation : Commandes Anchor CLI

Ces commandes couvrent le cycle de vie complet de developpement d'un programme Solana avec Anchor.

| Commande | Action | Usage typique |
|----------|--------|---------------|
| `anchor init my_program` | Creer un nouveau projet | Demarrage d'un nouveau programme |
| `anchor build` | Compiler le programme Rust | Apres modification du code |
| `anchor test` | Executer les tests TypeScript | Verification avant deploiement |
| `anchor deploy` | Deploier sur le cluster configure | Mise en production (devnet/mainnet) |
| `anchor run <script>` | Executer un script TypeScript | Operations one-off (mint, init, etc.) |

**Points cles** :
1. La structure de projet separe le code (`programs/`) des tests (`tests/`) et du frontend (`app/`)
2. `Anchor.toml` contient la configuration du cluster (localnet, devnet, mainnet) et les features du programme
3. Les tests sont ecrits en TypeScript (avec `@coral-xyz/anchor`) et interagissent avec le programme comme un client le ferait
4. `--skip-local-validator` permet de se connecter a un validator externe (devnet Solana publique) au lieu de lancer un validator local

**Workflow typique** :
```
anchor init mon_program    # Creer le projet
anchor build                # Compiler (verifier la syntaxe Rust)
anchor test                # Executer les tests (verifier la logique)
anchor deploy               # Deploier sur devnet
anchor run scripts/mint.ts  # Executer des operations d'initialisation
```

> **Note technique** : Anchor utilise `avm` (Anchor Version Manager) pour gerer les versions multiples. `avm install latest` installe la derniere version, `avm use latest` l'active.


---

## 7. Contributions etudiantes -- exemples guides

Les trois exemples ci-dessous, contribues par un groupe d'etudiants, illustrent les concepts fondamentaux de Solana et Anchor vus dans ce notebook : derivation de PDA, structure d'un programme Anchor, et modele de comptes Solana. Chaque enonce est suivi de sa solution commentee.


### Exemple guide 1 : Simulation Python de la derivation d'un PDA

_Exemple contribue par le groupe **Mehdi Azouz, Ryad Gazenay & Hani Bouzida** (PR #2189)._

Un PDA (Program Derived Address) est obtenu en cherchant un *bump* (255 -> 0) tel que le hash des seeds soit "hors courbe" ed25519 -- ici simule par la condition `premier octet < 128`. La derivation doit etre **deterministe** (memes seeds -> meme PDA) et **unique par utilisateur** (seeds differentes -> PDAs differents).


In [7]:
# Exercice 1 : Simulation Python de derivation PDA
import hashlib

def derive_pda_simulee(seeds: list, program_id: str) -> tuple:
    """
    Simule la derivation d'un PDA Solana en Python.

    Convention simplifiee : on cherche le premier bump (255..0) tel que
    SHA256(b"".join(seeds) + program_id.encode() + bytes([bump]))
    produise un hash dont le premier octet (int) est < 128
    (simule "hors courbe ed25519").

    Args:
        seeds     : liste de bytes (ex: [b"counter", b"alice_pubkey"])
        program_id: identifiant du programme (str)

    Returns:
        (pda_address: str, bump: int) ou (None, None) si aucun bump trouve
    """
    seeds_bytes = b"".join(seeds)
    for bump in range(255, -1, -1):
        data = seeds_bytes + program_id.encode() + bytes([bump])
        h = hashlib.sha256(data).hexdigest()
        if int(h[:2], 16) < 128:
            return h, bump
    return None, None

# --- Tests ---
seeds_alice = [b"counter", b"alice_pubkey_example"]
program_id  = "MyCounter1111111111111111111111111"

pda, bump = derive_pda_simulee(seeds_alice, program_id)
print(f"PDA alice        : {pda[:16]}...")
print(f"Bump             : {bump}")
pda2, _ = derive_pda_simulee(seeds_alice, program_id)
print(f"Deterministe     : {pda == pda2}")
seeds_bob = [b"counter", b"bob_pubkey_example"]
pda_bob, _ = derive_pda_simulee(seeds_bob, program_id)
print(f"PDA bob != alice : {pda != pda_bob}")


PDA alice        : 60ff9343476e7bb2...
Bump             : 255
Deterministe     : True
PDA bob != alice : True


### Exemple guide 2 : Programme Anchor de vote complet

_Exemple contribue par le groupe **Mehdi Azouz, Ryad Gazenay & Hani Bouzida** (PR #2189)._

Un programme Anchor de sondage : `create_poll` initialise le compte `Poll` (un PDA derive de `[b"poll", authority]`), et `vote` incremente le compteur du choix avec garde sur la fermeture du sondage et la validite du choix. Le code Rust complet est presente ci-dessous.


In [8]:
# Exercice 2 : Programme Anchor de Vote complete (Rust)

VOTE_PROGRAM_COMPLET = '''
use anchor_lang::prelude::*;

declare_id!("Vote1111111111111111111111111111111111111");

#[program]
pub mod voting {
    use super::*;

    pub fn create_poll(
        ctx: Context<CreatePoll>,
        question: String,
        option_a: String,
        option_b: String,
    ) -> Result<()> {
        let poll = &mut ctx.accounts.poll;
        poll.question  = question;
        poll.option_a  = option_a;
        poll.option_b  = option_b;
        poll.votes_a   = 0;
        poll.votes_b   = 0;
        poll.authority = ctx.accounts.authority.key();
        poll.is_open   = true;
        Ok(())
    }

    pub fn vote(ctx: Context<CastVote>, choice: u8) -> Result<()> {
        let poll = &mut ctx.accounts.poll;
        require!(poll.is_open, PollError::PollClosed);
        match choice {
            0 => poll.votes_a += 1,
            1 => poll.votes_b += 1,
            _ => return err!(PollError::InvalidChoice),
        }
        Ok(())
    }
}

#[derive(Accounts)]
pub struct CreatePoll<\'info> {
    #[account(
        init,
        payer = authority,
        space = 8 + Poll::INIT_SPACE,
        seeds = [b"poll", authority.key().as_ref()],
        bump
    )]
    pub poll: Account<\'info, Poll>,
    #[account(mut)]
    pub authority: Signer<\'info>,
    pub system_program: Program<\'info, System>,
}

#[derive(Accounts)]
pub struct CastVote<\'info> {
    #[account(mut, seeds = [b"poll", authority.key().as_ref()], bump)]
    pub poll: Account<\'info, Poll>,
    pub authority: Signer<\'info>,
}

#[account]
#[derive(InitSpace)]
pub struct Poll {
    pub authority: Pubkey,
    #[max_len(200)] pub question: String,
    #[max_len(100)] pub option_a: String,
    #[max_len(100)] pub option_b: String,
    pub votes_a: u64,
    pub votes_b: u64,
    pub is_open: bool,
}

#[error_code]
pub enum PollError {
    #[msg("Le sondage est ferme")] PollClosed,
    #[msg("Choix invalide : 0 ou 1 uniquement")] InvalidChoice,
}
'''

print("Exercice 2 - Programme de vote Anchor complete :")
print(VOTE_PROGRAM_COMPLET)


Exercice 2 - Programme de vote Anchor complete :

use anchor_lang::prelude::*;

declare_id!("Vote1111111111111111111111111111111111111");

#[program]
pub mod voting {
    use super::*;

    pub fn create_poll(
        ctx: Context<CreatePoll>,
        question: String,
        option_a: String,
        option_b: String,
    ) -> Result<()> {
        let poll = &mut ctx.accounts.poll;
        poll.question  = question;
        poll.option_a  = option_a;
        poll.option_b  = option_b;
        poll.votes_a   = 0;
        poll.votes_b   = 0;
        poll.authority = ctx.accounts.authority.key();
        poll.is_open   = true;
        Ok(())
    }

    pub fn vote(ctx: Context<CastVote>, choice: u8) -> Result<()> {
        let poll = &mut ctx.accounts.poll;
        require!(poll.is_open, PollError::PollClosed);
        match choice {
            0 => poll.votes_a += 1,
            1 => poll.votes_b += 1,
            _ => return err!(PollError::InvalidChoice),
        }
        Ok(())
  

### Exemple guide 3 : Modeliser un Counter Solana en Python

_Exemple contribue par le groupe **Mehdi Azouz, Ryad Gazenay & Hani Bouzida** (PR #2189)._

Pour valider la comprehension du **modele de comptes Solana**, on modelise en Python pur le programme Counter Anchor : `initialize` cree un compte `count = 0`, `increment`/`decrement` verifient l'authority avant toute modification, et `decrement` protege contre l'underflow (comme le ferait Anchor).


In [9]:
# Exercice 3 : Modeliser un Counter Solana en Python

class SolanaAccount:
    """Simule un compte Solana (programme owner + donnees)"""
    def __init__(self, owner: str, data: dict):
        self.owner = owner
        self.data = data

class CounterProgram:
    """Simule le programme Counter Anchor."""
    PROGRAM_ID = "Counter1111111111111111111111111111"

    def initialize(self, authority: str) -> SolanaAccount:
        return SolanaAccount(
            owner=self.PROGRAM_ID,
            data={"authority": authority, "count": 0}
        )

    def increment(self, counter: SolanaAccount, authority: str) -> None:
        if authority != counter.data["authority"]:
            raise ValueError("Autorite invalide")
        counter.data["count"] += 1

    def decrement(self, counter: SolanaAccount, authority: str) -> None:
        if authority != counter.data["authority"]:
            raise ValueError("Autorite invalide")
        if counter.data["count"] == 0:
            raise ValueError("Underflow")
        counter.data["count"] -= 1

# --- Tests ---
program = CounterProgram()
alice = "alice_pubkey_111111"

counter = program.initialize(alice)
print(f"Initialise  : count = {counter.data['count']}")   # attendu : 0
program.increment(counter, alice)
print(f"Increment   : count = {counter.data['count']}")   # attendu : 1
program.increment(counter, alice)
print(f"Increment   : count = {counter.data['count']}")   # attendu : 2
program.decrement(counter, alice)
print(f"Decrement   : count = {counter.data['count']}")   # attendu : 1
try:
    program.increment(counter, "intrus_pubkey_000")
except ValueError as e:
    print(f"Autorite invalide interceptee : {e}")


Initialise  : count = 0
Increment   : count = 1
Increment   : count = 2
Decrement   : count = 1
Autorite invalide interceptee : Autorite invalide


### Exercice (a completer) : Etendre le Counter avec un plafond et un reset

A votre tour ! En vous inspirant de l'exemple guide 3, faites evoluer la classe `CounterProgram` :

1. Ajoutez une borne `MAX_COUNT = 10` : `increment` doit lever une exception (interceptee) si l'on depasse le plafond.
2. Ajoutez une methode `reset(counter, authority)` qui remet `count` a 0 -- **seule** l'authority du compte peut la declencher.
3. Ecrivez un petit scenario de test qui incremente jusqu'au plafond, tente un increment de trop (try/except), puis effectue un reset.

**Indice** : reprenez le pattern de verification `authority != counter.data["authority"]` deja utilise dans `increment`/`decrement`.


In [10]:
# Exercice : etendre CounterProgram avec MAX_COUNT (plafond) et reset(counter, authority)
# TODO etudiant :
#   Etape 1 : sous-classer ou modifier CounterProgram avec MAX_COUNT = 10
#   Etape 2 : increment leve une exception (interceptee) si count atteindrait le plafond
#   Etape 3 : reset(counter, authority) remet count a 0, reserve a l'authority
#   Etape 4 : scenario de test (increment jusqu'au plafond, increment de trop, reset)

print("Exercice a completer")


Exercice a completer


---

## 8. Resume

| Concept | Description |
|---------|-------------|
| Program | Code executable stateless |
| Account | Stockage de donnees |
| PDA | Adresse derivee du programme |
| CPI | Appel inter-programmes |
| Anchor | Framework Rust pour Solana |

---

**Notebook suivant** : [SC-23-Cross-Chain](../06-Real-World/SC-23-Cross-Chain.ipynb)

---

[<< Move & Sui](SC-21-Move-Sui.ipynb) | [Cross-Chain >>](../06-Real-World/SC-23-Cross-Chain.ipynb)

### Interpretation : Token Escrow avec CPI

Cet exercice presente un escrow atomique (echange de tokens sans tiers de confiance) utilisant des CPIs.

| Fonction | Operation | CPI utilise |
|----------|-----------|-------------|
| `make` | Initialise l'escrow et transfere Token A vers vault | `token::transfer` (initializer → vault) |
| `take` | Echange atomique : Token B → initializer, Token A → depositor | 2x `token::transfer` (depositor → initializer, vault → depositor) |

**Points cles** :
1. `make` cree un compte escrow avec seeds `[b"escrow", maker.key(), seed]` (PDA deterministe)
2. Le vault est un compte token PDA controle par l'escrow (pas de cle privee)
3. `take` utilise `CpiContext::new_with_signer` avec les seeds de l'escrow PDA pour autoriser le transfert depuis le vault
4. L'echange est atomique : soit les 2 transferts reussissent, soit aucun (revert si l'un echoue)

**Securite** :
- Le seed `u64` permet a un utilisateur d'avoir plusieurs escrows simultanes
- Le bump `ctx.bumps.escrow` est inclus dans la signature CPI pour prouver l'autorite du PDA
- L'ordre des transferts est important : d'abord le token B du depositor, puis le token A du vault (evite certains attaques)

> **Note technique** : Sur Solana, les escrows sont des PDAs (pas de contrats intelligents avec etat). Le PDA signe les transferts via `new_with_signer`, ce qui est impossible sur EVM.


## Resume et perspectives

Ce notebook a permis de decourir l'ecosysteme Solana et le framework Anchor comme alternative a l'ecosysteme Ethereum/Solidity. Nous avons compare les architectures : modele de comptes stateless et execution BPF haute performance (~65 000 TPS theoriques) chez Solana versus stockage dans le contrat et EVM chez Ethereum. Nous avons implemente un programme Counter en Rust avec Anchor, explorant les annotations `#[program]`, `#[account]` et `#[derive(Accounts)]` qui abstraient la complexite du modele de comptes. Nous avons etudie les PDAs (Program Derived Addresses) pour le stockage deterministe et securise de donnees utilisateur, les CPIs (Cross-Program Invocations) pour la composition de programmes, et implémente un escrow atomique de tokens SPL combinant `CpiContext::new` et `CpiContext::new_with_signer` pour les transfers signes par PDA.

La difference philosophique majeure entre Solana et Ethereum reside dans la separation stricte entre code (programmes stateless) et donnees (comptes independants). Ce modele enable le parallelisme natif des transactions et elimine la contention globale, mais exige une planification plus rigoureuse de la structure des comptes. Les PDAs remplacent les mappings Solidity par des adresses deterministes sans cle privee, offrant une securite inherente contre les acces non autorises. Le framework Anchor simplifie considerablement le developpement en Rust tout en preservant les garanties de securite du type system.

Le notebook suivant, [SC-23-Cross-Chain](../06-Real-World/SC-23-Cross-Chain.ipynb), aborde l'interoperabilite entre blockchains et les protocols de communication cross-chain comme Chainlink CCIP.